# Model Training and Model Selection

This notebook trains and compares several regression models to predict median house values using the California Housing Prices dataset.

Following the evaluation strategy defined in the business understanding notebook:

- A simple **baseline** (predicting the training mean) is established first.
- **Linear Regression** is the first real model and the reference point for more complex models.
- More flexible models (**Random Forest** and **Histogram Gradient Boosting**) are compared against both.
- Model selection is done with **5-fold cross-validation on the training set only**. The held-out test set is not used to choose a model.
- The selected model is refitted on the full training set, and predictions for all models are saved for the evaluation notebook.

**Dataset limitation:** The dataset is based on historical 1990 California census data, and the target variable contains values capped at $500,001. These limitations are considered when interpreting results.

## 1. Load Prepared Data

The prepared training and test datasets, target variables, and fitted preprocessing pipeline are loaded from the artifacts directory created in the data preparation notebook.

In [15]:
import os
import joblib
import numpy as np
import pandas as pd

artifacts_dir = "../artifacts"

X_train = joblib.load(os.path.join(artifacts_dir, "X_train_prepared.joblib"))
X_test = joblib.load(os.path.join(artifacts_dir, "X_test_prepared.joblib"))
y_train = joblib.load(os.path.join(artifacts_dir, "y_train.joblib"))
y_test = joblib.load(os.path.join(artifacts_dir, "y_test.joblib"))

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

Training features: (16512, 16)
Test features: (4128, 16)
Training target: (16512,)
Test target: (4128,)


## 2. Define Candidate Models

Four candidates are defined:

- **Dummy (mean):** always predicts the mean of the training target. Any useful model must beat it.
- **Linear Regression:** simple, interpretable, and unable to capture non-linear patterns or interactions.
- **Random Forest:** an ensemble of decision trees that can capture non-linear relationships and interactions such as the combined effect of latitude and longitude.
- **Histogram Gradient Boosting:** a boosted tree ensemble that is fast on tabular data and often among the strongest models for this type of problem.

All models use `random_state=42` where applicable, so results are reproducible. Default hyperparameters are used at this stage; tuning is a later step.

In [16]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

models = {
    "Dummy (mean)": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
}

list(models.keys())

['Dummy (mean)',
 'Linear Regression',
 'Random Forest',
 'Hist Gradient Boosting']

## 3. Compare Models with Cross-Validation

Each model is evaluated with 5-fold cross-validation on the **training set only**. For each fold, MAE, MSE, RMSE, and R² are computed on the validation fold, and the mean and standard deviation across folds are reported.

MAE is the primary metric, as defined in the business understanding notebook.

**Note:** The preprocessing pipeline was fitted on the full training set before cross-validation, so the imputation medians and scaling parameters were learned from all training folds. This is a small simplification that has a negligible effect on the comparison, and the test set is not involved at any point.

In [17]:
from sklearn.model_selection import KFold, cross_validate

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "mae": "neg_mean_absolute_error",
    "mse": "neg_mean_squared_error",
    "r2": "r2",
}

cv_rows = []
for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train, cv=kfold, scoring=scoring, n_jobs=1
    )
    mae = -scores["test_mae"]
    mse = -scores["test_mse"]
    rmse = np.sqrt(mse)
    r2 = scores["test_r2"]
    cv_rows.append({
        "model": name,
        "MAE (mean)": mae.mean(),
        "MAE (std)": mae.std(),
        "MSE (mean)": mse.mean(),
        "RMSE (mean)": rmse.mean(),
        "R² (mean)": r2.mean(),
        "R² (std)": r2.std(),
    })

cv_results = pd.DataFrame(cv_rows).set_index("model")
cv_results.round(4)

,MAE (mean),MAE (std),MSE (mean),RMSE (mean),R² (mean),R² (std)
model,,,,,,
Dummy (mean),91391.4745,1008.2429,1.336881e+10,115617.1853,-0.0002,0.0002
Linear Regression,48879.9088,982.3483,4.606536e+09,67858.7785,0.6552,0.0143
Random Forest,32910.5595,529.4744,2.530923e+09,50297.6752,0.8107,0.0049
Hist Gradient Boosting,31201.8619,540.9195,2.201770e+09,46910.3259,0.8353,0.0050


In [18]:
display_cols = ["MAE (mean)", "MAE (std)", "RMSE (mean)", "R² (mean)"]
cv_results[display_cols].round(2)

,MAE (mean),MAE (std),RMSE (mean),R² (mean)
model,,,,
Dummy (mean),91391.47,1008.24,115617.19,-0.00
Linear Regression,48879.91,982.35,67858.78,0.66
Random Forest,32910.56,529.47,50297.68,0.81
Hist Gradient Boosting,31201.86,540.92,46910.33,0.84


### Observations

- The Dummy model has a cross-validation MAE of about $91,400 and an R² of about 0. This is the level any useful model must beat.
- Linear Regression reduces the MAE to about $48,900 (R² ≈ 0.66), which confirms that the features carry real signal but that a linear model leaves a lot unexplained.
- Random Forest (MAE ≈ $32,900, R² ≈ 0.81) and Histogram Gradient Boosting (MAE ≈ $31,200, R² ≈ 0.83) are clearly better than Linear Regression. The tree-based models can capture non-linear effects and interactions, such as the combined effect of location and income.
- The standard deviation of MAE across folds is small (about $500 to $1,000), so the gaps between Dummy, Linear Regression, and the tree-based models are much larger than the fold-to-fold variation. The gap between Random Forest and Gradient Boosting (about $1,700) is smaller but still larger than their fold variation.
- Histogram Gradient Boosting has the lowest MAE and the lowest RMSE, and it is selected.

## 4. Select the Best Model

The model with the lowest mean cross-validation MAE is selected. The Dummy model is excluded from selection because it is only a reference baseline.

In [19]:
candidates = cv_results.drop(index="Dummy (mean)")
best_model_name = candidates["MAE (mean)"].idxmin()

print("Selected model:", best_model_name)
print(f"CV MAE: ${candidates.loc[best_model_name, 'MAE (mean)']:,.2f}")

Selected model: Hist Gradient Boosting
CV MAE: $31,201.86


## 5. Fit All Models on the Full Training Set and Predict the Test Set

Every model is refitted on the full training set. Predictions on the test set are generated for all models and saved, so the evaluation notebook can compare them on the same held-out data. The test set is used here only for prediction, not for any selection decision.

In [20]:
test_predictions = {}
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    test_predictions[name] = model.predict(X_test)

print("First 5 predictions of the selected model:")
print(test_predictions[best_model_name][:5].round(0))

First 5 predictions of the selected model:
[ 59379.  76510. 470351. 249354. 242523.]


## 6. Save Artifacts

The selected model, the test predictions of every model, the name of the selected model, and the cross-validation results are saved for the evaluation notebook.

In [21]:
os.makedirs(artifacts_dir, exist_ok=True)

joblib.dump(fitted_models[best_model_name], os.path.join(artifacts_dir, "best_model.joblib"))
joblib.dump(test_predictions, os.path.join(artifacts_dir, "test_predictions.joblib"))
joblib.dump(best_model_name, os.path.join(artifacts_dir, "best_model_name.joblib"))
joblib.dump(cv_results, os.path.join(artifacts_dir, "cv_results.joblib"))

print(sorted(os.listdir(artifacts_dir)))

['X_test_prepared.joblib', 'X_train_prepared.joblib', 'best_model.joblib', 'best_model_name.joblib', 'cv_results.joblib', 'preprocessor.joblib', 'test_predictions.joblib', 'y_test.joblib', 'y_train.joblib']
